In [0]:
print("changed notebook name")

# PySpark intro

PySpark is the Python API to Apache Spark

A short note on notebooks in Databricks, can choose between:
- SQL notebook
- Python notebook


In [0]:
from pathlib import Path
print("hej")

In [0]:
DATA_PATH = "/Volumes/data/olympic_games/raw_data"


df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True)
df_athletes

In [0]:
df_athletes.limit(2).display()

In [0]:
(df_athletes.count)
 # missing something here

# Infer the schema

In [0]:

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, inferSchema=True)
df_athletes

In [0]:
display(df_athletes)

# Define explicit schema

In [0]:
df_athletes_schema = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
display(df_athletes_schema)

In [0]:
from pyspark.sql.types import StructField,3 StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", ShortType()),
  StructField("Weight", ShortType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

# EDA

# PySpark transformations

In [0]:
df_athletes_schema.groupBy("NOC", "Medal").count().filter(
    "NOC IN ('SWE', 'NOR', 'FIN', 'DEN', 'ISL') AND Medal != 'NA'"
).sort("NOC", "Medal").display()
     

# Spark SQL

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")
spark.sql("FROM df_athletes_schema").display()

In [0]:
spark.sql(
    """
          SELECT
            name, 
            sport,
            event,
            medal,
            year
          FROM df_athletes_schema
          WHERE 
            sport = 'Table Tennis' AND
            medal != 'NA' AND
            noc = 'SWE'
          """
).display()

# Count medals and plot

In [0]:
df_swe_medals = spark.sql("""
          SELECT
            sport, count(medal) AS medal_count
          FROM df_athletes_schema
          WHERE noc = 'SWE' AND medal IN ('Bronze', 'Silver','Gold')
          GROUP BY sport 
          ORDER BY medal_count DESC
          """)

df_swe_medals.display()
     

In [0]:
df_swe_medals.plot(kind = "bar", y="sport", x = "medal_count")

# Ingesting data to Unity Catalog

In [0]:
%sql
SHOW SCHEMAS IN data;

CREATE TABLE IF NOT EXISTS data.olympic_games.sweden_medals AS 
(
    SELECT 
        name, age, height, weight,year, sport, medal
    FROM df_athletes_schema
    WHERE NOC = 'SWE' AND medal IN ('Bronze', 'Silver', 'Gold')
)

In [0]:
%sql
SELECT 
name, sport, medal 
FROM data.olympic_games.sweden_medals
WHERE sport = 'Swimming'